# Figures S1J / S1K — Time-Dependent Validation

**S1J — Time-integrated Brier disagreement between LLM- and gold-standard time-to-event endpoints.** For each toxicity, the LLM pipeline and gold standard are treated as two binary event-by-time-*t* trajectories. At each time point, a line contributes only when the status is observable in **both** arms (event has occurred by *t*, or the shared follow-up remains uncensored past *t*). The Brier contribution is therefore 0 for agreement and 1 for disagreement. The score is integrated to 12 months; lower is better.

**S1K — Onset agreement among concordant events.** Restricted to lines where both the gold standard and the pipeline flagged the toxicity, so both arms carry identical n. This isolates timing; sensitivity is reported separately.

**Backbone (censoring, `lot_start`, `cancer_type`):** per-toxicity master tables (`llm84k_{toxicity}_grade0_20260630.csv`); `t_cutoff_lot` used directly as `censor_days`. Scaffolding only, not the outcome being validated.

**Gold standard:** `2026May01_merged_ae_with_apr_full.csv`.

**LLM side:** `llama_maverick_84k_patient_results.csv`, with the first threshold-positive batch defining `llm_onset_days`.

## Brier definition

At each time *t*, let `Y_GS(t)` and `Y_LLM(t)` be the binary event-by-*t* status. A line is scored only if both statuses are observable at *t*. The time-specific score is:

`BS(t) = mean[(Y_LLM(t) - Y_GS(t))^2]`.

Because both outcomes are binary, this is exactly the proportion of observable lines on which the two endpoint trajectories disagree at time *t*. The reported statistic is the time-integrated Brier score (IBS) over 0–12 months.

## Outputs
- `Brier_Score_S1J1.pdf` + `Brier_Score_S1J1_results.csv` — time-integrated Brier disagreement, patient-clustered 95% bootstrap CIs
- `KM_Concordant_S1K.pdf` + `Onset_Agreement_S1J2_results.csv` — onset agreement, all six toxicities
- `Onset_Timing_Error_S1K.csv` — median absolute onset error (days/weeks) with bootstrap 95% CIs

**Formatting:** Arial only (hard-fails if unresolved), `pdf.fonttype=42`, `dpi=450`.


In [ ]:
import os
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from lifelines import KaplanMeierFitter
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import GroupKFold
from scipy.stats import wilcoxon

%matplotlib inline

warnings.filterwarnings('ignore')

# ---- Style block matched to Grade_Diff_S1I.ipynb (font, sizes, linewidths) ----
plt.rcParams.update({
    "font.family": "Arial",
    "font.size": 6,
    "axes.labelsize": 7,
    "xtick.labelsize": 6,
    "ytick.labelsize": 6,
    "axes.linewidth": 0.6,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

# ---- Hard-fail if Arial isn't actually resolved (no silent fallback) ----
import matplotlib.font_manager as fm
_arial_path = fm.findfont('Arial', fallback_to_default=False)
if 'Arial' not in _arial_path:
    raise RuntimeError(
        f"Arial not found -- matplotlib resolved to '{_arial_path}' instead. "
        "Install Arial or update font.sans-serif before rendering this figure."
    )
print(f"Arial resolved to: {_arial_path}")


## Paths

In [ ]:
NOTEBOOK_DIR = os.getcwd()
# ---------------------------------------------------------------------------
# FILE PATHS — notebook lives in figure 1/scripts/
#   figures/
#   ├── figures_data/figure 1/data/   ← inputs (shared OneDrive data dir)
#   └── v1/figure 1/
#       ├── scripts/                  ← this notebook
#       └── results/supp/S1J_time_dependent_validation/
# ---------------------------------------------------------------------------
from pathlib import Path
ROOT = Path("..").resolve()
FIGURES = ROOT.parent.parent
DATA_DIR = str(FIGURES / "figures_data" / "figure 1" / "data")
TOX_TABLE_DIR = os.path.join(DATA_DIR, 'OneDrive_1_8-7-2026')
RESULTS_DIR = str(ROOT / "results" / "supp" / "S1J_time_dependent_validation")
os.makedirs(RESULTS_DIR, exist_ok=True)

_gs_candidates = [
    os.path.join(DATA_DIR, '2026may01_merged_ae_with_apr_full.csv'),
    os.path.join(DATA_DIR, '2026May01_merged_ae_with_apr_full.csv'),
]
GS_PATH = next((p for p in _gs_candidates if os.path.exists(p)), _gs_candidates[0])
RAW_PROB_PATH = os.path.join(DATA_DIR, 'llama_maverick_84k_patient_results.csv')

BRIER_PDF_OUT = os.path.join(RESULTS_DIR, 'Brier_Score_S1J1.pdf')
BRIER_CSV_OUT = os.path.join(RESULTS_DIR, 'Brier_Score_S1J1_results.csv')
KM_PDF_OUT = os.path.join(RESULTS_DIR, 'KM_Concordant_S1K.pdf')
KM_CSV_OUT = os.path.join(RESULTS_DIR, 'Onset_Agreement_S1J2_results.csv')
TIMING_CSV_OUT = os.path.join(RESULTS_DIR, 'Onset_Timing_Error_S1K.csv')
CALIB_CSV_OUT = os.path.join(RESULTS_DIR, 'Brier_Score_S1J1_calibrated.csv')

TOXICITY_COLUMNS = ['pneumonitis', 'adrenal_insufficiency', 'liver_toxicity',
                    'colitis', 'hyperthyroidism', 'hypothyroidism']
TOXICITY_DISPLAY = {
    'pneumonitis': 'Pneumonitis', 'adrenal_insufficiency': 'Adrenal Insufficiency',
    'liver_toxicity': 'Liver Toxicity', 'colitis': 'Colitis',
    'hyperthyroidism': 'Hyperthyroidism', 'hypothyroidism': 'Hypothyroidism',
}
TOXICITY_MAP = {
    'pneumonitis': 'pneumonitis',
    'adrenal insufficiency': 'adrenal_insufficiency',
    'adrenal_insufficiency': 'adrenal_insufficiency',
    'liver toxicity': 'liver_toxicity',
    'liver_toxicity': 'liver_toxicity',
    'colitis': 'colitis',
    'hyperthyroidism': 'hyperthyroidism',
    'hypothyroidism': 'hypothyroidism',
}
GRADE_TIER = 'grade0'  # == Grade 1+, confirmed empirically
TOX_TABLE_PATHS = {
    tox: os.path.join(TOX_TABLE_DIR, f'llm84k_{tox}_{GRADE_TIER}_20260630.csv')
    for tox in TOXICITY_COLUMNS
}

print(f'DATA_DIR     : {DATA_DIR}')
print(f'RESULTS_DIR  : {RESULTS_DIR}')
_missing = []
for p in list(TOX_TABLE_PATHS.values()) + [GS_PATH, RAW_PROB_PATH]:
    ok = os.path.exists(p)
    print(('FOUND   ' if ok else 'MISSING '), os.path.basename(p))
    if not ok:
        _missing.append(p)
if _missing:
    raise FileNotFoundError(
        'Required input(s) not found:\n  ' + '\n  '.join(_missing)
    )


## Constants

In [ ]:
AE_THRESHOLDS = {
    'pneumonitis': 0.710,
    'adrenal_insufficiency': 0.810,
    'liver_toxicity': 0.010,
    'colitis': 0.710,
    'hyperthyroidism': 0.810,
    'hypothyroidism': 0.510,
}

T_MONTHS = 12.0

def standardize_mrn(mrn):
    if pd.isna(mrn):
        return None
    try:
        digits = re.findall(r'\d+', str(mrn).strip().strip("'\""))
        return str(int(digits[0])).zfill(8) if digits else None
    except (ValueError, TypeError):
        return None


## Backbone: every (patient, LOT), with pre-computed censoring

In [ ]:
backbone_raw = pd.read_csv(TOX_TABLE_PATHS['pneumonitis'], low_memory=False)
backbone_raw['mrn'] = backbone_raw['mrn'].apply(standardize_mrn)
backbone_raw = backbone_raw[backbone_raw['mrn'].notna()].copy()
backbone_raw['lot_start'] = pd.to_datetime(backbone_raw['lot_start'], errors='coerce')

backbone = backbone_raw.copy()
backbone['censor_days'] = pd.to_numeric(backbone['t_cutoff_lot'], errors='coerce')
backbone = backbone[np.isfinite(backbone['censor_days']) & (backbone['censor_days'] > 0)].copy()
backbone = backbone[['mrn', 'lot', 'lot_start', 'cancer_type', 'censor_days']].rename(columns={'lot': 'lot_clean'})
backbone['lot_clean'] = pd.to_numeric(backbone['lot_clean'], errors='coerce')  # <-- add this line
backbone = backbone[backbone['lot_clean'].notna()].copy()  # <-- and this, drop any that failed to parse
print(f'{len(backbone):,} LOT-level records with valid (finite, positive) censoring, '
      f'{backbone["mrn"].nunique():,} patients')

## Gold standard — matched per line of therapy via `MRN` + `APR_LOT`

`Toxicity` is an exact Figure 1B `TOXICITY_MAP` lookup (`pneumonitis`, `adrenal insufficiency`, `liver toxicity`, `colitis`, `hyperthyroidism`, `hypothyroidism`). Synonyms (`hepatitis`, `addison`, `thyrotoxicosis`, …) are not expanded.

Rows with non-numeric `APR_LOT` (`"10+"`, `"No Match"`, blank) can't be tied to a specific line
and are excluded (~27% of the file). Audited: this exclusion accounts for essentially none of the
GS/LLM incidence gap (see audit trail at the end).

In [ ]:
gs = pd.read_csv(GS_PATH, encoding='latin-1', low_memory=False)
gs['mrn'] = gs['MRN'].apply(standardize_mrn)
gs = gs[gs['mrn'].notna()].copy()

gs['lot_numeric'] = pd.to_numeric(gs['APR_LOT'], errors='coerce')
n_total = len(gs)
n_excluded = gs['lot_numeric'].isna().sum()
print(f'Excluding {n_excluded:,} / {n_total:,} rows ({100*n_excluded/n_total:.1f}%) with '
      f'non-numeric APR_LOT (10+, No Match, blank, etc.)')
gs = gs[gs['lot_numeric'].notna()].copy()

gs['Start Date'] = pd.to_datetime(gs['Start Date'], errors='coerce')
gs = gs[gs['Start Date'].notna()].copy()

gs['tox_lower'] = gs['Toxicity'].fillna('').astype(str).str.lower().str.strip()
gs['tox_mapped'] = gs['tox_lower'].map(TOXICITY_MAP)
gs_mapped = gs[gs['tox_mapped'].notna()].copy()
print(f'{len(gs_mapped):,} mapped AE rows across {gs_mapped["mrn"].nunique():,} patients')

# Match to the specific (mrn, lot) row in the backbone
gs_mapped = gs_mapped.merge(
    backbone[['mrn', 'lot_clean', 'lot_start', 'censor_days']],
    left_on=['mrn', 'lot_numeric'], right_on=['mrn', 'lot_clean'], how='inner'
)
gs_mapped['days_from_lot_start'] = (gs_mapped['Start Date'] - gs_mapped['lot_start']).dt.days

n_negative = (gs_mapped['days_from_lot_start'] < 0).sum()
print(f'Excluding {n_negative:,} GS records with AE date before this specific line\'s start')
gs_mapped = gs_mapped[gs_mapped['days_from_lot_start'] >= 0].copy()
gs_mapped = gs_mapped[gs_mapped['days_from_lot_start'] <= gs_mapped['censor_days']].copy()
print(f'{len(gs_mapped):,} GS AE records matched within their line\'s censoring window')


### Restrict to gold-standard-reviewed patients

Absence from the GS file means *never chart-reviewed*, not *confirmed AE-free*. Without this,
~96,000 lines carry fabricated negative ground truth.

In [ ]:
gs_reviewed_mrns = set(gs['mrn'].dropna())
print(f'{len(gs_reviewed_mrns):,} patients with any record in the GS file (reviewed cohort)')

backbone = backbone[backbone['mrn'].isin(gs_reviewed_mrns)].copy()
print(f'{len(backbone):,} LOT-level records after restricting to GS-reviewed patients, '
      f'{backbone["mrn"].nunique():,} patients')

## Build GS (time, event) per toxicity

In [ ]:
def build_gs_survival(tox):
    tox_records = gs_mapped[gs_mapped['tox_mapped'] == tox]
    first_onset = (tox_records.groupby(['mrn', 'lot_clean'])['days_from_lot_start']
                   .min().reset_index().rename(columns={'days_from_lot_start': 'gs_onset_days'}))
    surv = backbone[['mrn', 'lot_clean', 'censor_days']].merge(
        first_onset, on=['mrn', 'lot_clean'], how='left')
    surv['event'] = surv['gs_onset_days'].notna().astype(int)
    surv['time'] = np.where(surv['event'] == 1, surv['gs_onset_days'], surv['censor_days'])
    surv = surv[surv['time'] > 0].copy()
    return surv.reset_index(drop=True)

gs_survival = {tox: build_gs_survival(tox) for tox in TOXICITY_COLUMNS}
for tox in TOXICITY_COLUMNS:
    s = gs_survival[tox]
    print(f'  {TOXICITY_DISPLAY[tox]}: n={len(s):,} lines, events={s["event"].sum():,}')


## LLM side — raw probabilities, matched per line of therapy

In [ ]:
raw_prob = pd.read_csv(RAW_PROB_PATH, encoding='latin-1', low_memory=False)
raw_prob = raw_prob.rename(columns={
    'liver toxicity': 'liver_toxicity', 'adrenal insufficiency': 'adrenal_insufficiency',
})
raw_prob['mrn'] = raw_prob['mrn'].apply(standardize_mrn)
raw_prob = raw_prob[raw_prob['mrn'].notna()].copy()
raw_prob['window_start'] = pd.to_datetime(raw_prob['window_start'], errors='coerce')
raw_prob = raw_prob[raw_prob['window_start'].notna()].copy()
for tox in TOXICITY_COLUMNS:
    if tox in raw_prob.columns:
        raw_prob[tox] = pd.to_numeric(raw_prob[tox], errors='coerce').fillna(0.0)
print(f'{len(raw_prob):,} probability-scored batches for {raw_prob["mrn"].nunique():,} patients')

prob_matched = raw_prob.merge(backbone[['mrn', 'lot_clean', 'lot_start', 'censor_days']], on='mrn', how='inner')
prob_matched['days_from_lot_start'] = (prob_matched['window_start'] - prob_matched['lot_start']).dt.days
prob_in_window = prob_matched[(prob_matched['days_from_lot_start'] >= 0) &
                               (prob_matched['days_from_lot_start'] <= prob_matched['censor_days'])].copy()
print(f'{len(prob_in_window):,} batch-line matches within their line\'s censoring window')

def build_llm_max_prob(tox):
    return prob_in_window.groupby(['mrn', 'lot_clean'])[tox].max()

def build_llm_survival(tox):
    thr = AE_THRESHOLDS[tox]
    pos = prob_in_window[prob_in_window[tox] >= thr]
    first_pos = pos.groupby(['mrn', 'lot_clean'])['days_from_lot_start'].min().reset_index().rename(
        columns={'days_from_lot_start': 'llm_onset_days'})
    surv = backbone[['mrn', 'lot_clean', 'censor_days']].merge(first_pos, on=['mrn', 'lot_clean'], how='left')
    surv['event'] = surv['llm_onset_days'].notna().astype(int)
    surv['time'] = np.where(surv['event'] == 1, surv['llm_onset_days'], surv['censor_days'])
    surv = surv[surv['time'] > 0].copy()
    return surv.reset_index(drop=True)

llm_max_prob = {tox: build_llm_max_prob(tox) for tox in TOXICITY_COLUMNS}
llm_survival = {tox: build_llm_survival(tox) for tox in TOXICITY_COLUMNS}
for tox in TOXICITY_COLUMNS:
    s = llm_survival[tox]
    print(f'  {TOXICITY_DISPLAY[tox]}: LLM events={s["event"].sum():,} (thr={AE_THRESHOLDS[tox]})')


## Joint-observability audit

The Brier comparison is restricted to time points where **both** endpoint statuses are observable. A status is observable at time *t* if the endpoint has already occurred by *t*, or if that arm's censoring/follow-up time extends beyond *t*. If one arm is censored before *t* and has not had an event, that line is excluded at that time point rather than counted as a disagreement.


In [ ]:
# ============ DIAGNOSTIC: jointly observable set across the time grid ============
print('Joint-observability audit (pneumonitis, representative):')
_g = gs_survival['pneumonitis'].set_index(['mrn', 'lot_clean'])
_l = llm_survival['pneumonitis'].set_index(['mrn', 'lot_clean'])
_a = _g.join(_l[['llm_onset_days', 'censor_days']].rename(columns={'censor_days': 'llm_censor_days'}), how='inner')
for _t in [30, 90, 180, 270, 365]:
    _gs_obs = _a['gs_onset_days'].notna() & (_a['gs_onset_days'] <= _t) | (_a['censor_days'] > _t)
    _llm_obs = _a['llm_onset_days'].notna() & (_a['llm_onset_days'] <= _t) | (_a['llm_censor_days'] > _t)
    _mask = _gs_obs & _llm_obs
    print(f'  t={_t:3d}d  joint-observable={_mask.sum():6,}  excluded={len(_a)-_mask.sum():6,}')


## Time-integrated Brier disagreement

Both the GS and LLM endpoint are binary event-by-time-*t* trajectories. Because the two outcomes are binary, the squared error is simply an agreement/disagreement indicator. Censored observations are excluded at time points where their status is not yet known. Patient-level bootstrap resampling is used for 95% CIs.


In [ ]:
def _integrated_brier_endpoint_agreement(gs_times, gs_events, gs_censor,
                                           llm_times, llm_events, llm_censor,
                                           time_grid):
    """Time-integrated Brier disagreement between two censored binary event-time endpoints.

    At each t, each endpoint is observable if it has an event by t OR its censoring/follow-up
    time is still beyond t. Only lines observable in both arms contribute. For binary endpoints,
    (Y_LLM - Y_GS)^2 is 0 for agreement and 1 for disagreement.
    """
    gs_times = np.asarray(gs_times, float)
    gs_events = np.asarray(gs_events, int)
    gs_censor = np.asarray(gs_censor, float)
    llm_times = np.asarray(llm_times, float)
    llm_events = np.asarray(llm_events, int)
    llm_censor = np.asarray(llm_censor, float)

    valid = (np.isfinite(gs_times) & np.isfinite(gs_censor) &
             np.isfinite(llm_times) & np.isfinite(llm_censor))
    if not valid.any():
        return np.nan, 0, np.array([]), np.array([])

    gs_times, gs_events, gs_censor = gs_times[valid], gs_events[valid], gs_censor[valid]
    llm_times, llm_events, llm_censor = llm_times[valid], llm_events[valid], llm_censor[valid]

    bs, vt, n_obs = [], [], []
    for t in time_grid:
        gs_obs = ((gs_events == 1) & (gs_times <= t)) | (gs_censor > t)
        llm_obs = ((llm_events == 1) & (llm_times <= t)) | (llm_censor > t)
        mask = gs_obs & llm_obs
        if not mask.any():
            continue

        y_gs = ((gs_events == 1) & (gs_times <= t))[mask].astype(int)
        y_llm = ((llm_events == 1) & (llm_times <= t))[mask].astype(int)
        bs.append(np.mean((y_llm - y_gs) ** 2))
        vt.append(t)
        n_obs.append(int(mask.sum()))

    if len(bs) < 2:
        return np.nan, len(bs), np.asarray(vt), np.asarray(n_obs)
    _trapz = getattr(np, 'trapezoid', None) or np.trapz
    rng_t = vt[-1] - vt[0]
    if rng_t <= 0:
        return np.nan, len(vt), np.asarray(vt), np.asarray(n_obs)
    ibs = _trapz(bs, vt) / rng_t
    return float(max(0.0, min(1.0, ibs))), len(vt), np.asarray(vt), np.asarray(n_obs)


TIME_GRID_DAYS = np.linspace(0, T_MONTHS * 30.44, 50)
N_BOOTSTRAP, RNG_SEED = 500, 0

brier_records = []
for tox in TOXICITY_COLUMNS:
    gs_s = gs_survival[tox].set_index(['mrn', 'lot_clean'])
    llm_s = llm_survival[tox].set_index(['mrn', 'lot_clean'])

    aligned = gs_s[['gs_onset_days', 'event', 'time', 'censor_days']].join(
        llm_s[['llm_onset_days', 'event', 'time', 'censor_days']].rename(columns={
            'event': 'llm_event', 'time': 'llm_time', 'censor_days': 'llm_censor_days'
        }),
        how='inner'
    )

    point_est, n_grid, used_t, n_obs = _integrated_brier_endpoint_agreement(
        aligned['time'].values, aligned['event'].values, aligned['censor_days'].values,
        aligned['llm_time'].values, aligned['llm_event'].values, aligned['llm_censor_days'].values,
        TIME_GRID_DAYS
    )

    # Bootstrap PATIENTS (not lines) to respect within-patient clustering.
    groups = aligned.index.get_level_values('mrn').values
    uniq = np.unique(groups)
    idx_by_pt = {g: np.where(groups == g)[0] for g in uniq}
    rng = np.random.default_rng(RNG_SEED)
    boot = np.empty(N_BOOTSTRAP)
    for b in range(N_BOOTSTRAP):
        pick = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([idx_by_pt[g] for g in pick])
        boot[b] = _integrated_brier_endpoint_agreement(
            aligned['time'].values[idx], aligned['event'].values[idx], aligned['censor_days'].values[idx],
            aligned['llm_time'].values[idx], aligned['llm_event'].values[idx], aligned['llm_censor_days'].values[idx],
            TIME_GRID_DAYS
        )[0]
    ci_lower, ci_upper = np.nanpercentile(boot, [2.5, 97.5])

    # Agreement score is included only as an intuitive complement: 1 - IBS, higher is better.
    agreement = 1.0 - point_est if np.isfinite(point_est) else np.nan

    brier_records.append({
        'toxicity': tox,
        'toxicity_display': TOXICITY_DISPLAY[tox],
        'time_integrated_brier': point_est,
        'agreement_score_1_minus_brier': agreement,
        'ci_95_lower': ci_lower,
        'ci_95_upper': ci_upper,
        'n_grid_points_used': n_grid,
        'n_lines': len(aligned),
        'n_patients': len(uniq),
        'n_gs_events': int(aligned['event'].sum()),
        'n_llm_events': int(aligned['llm_event'].sum()),
        'median_gs_censor_days': float(aligned['censor_days'].median()),
        'median_llm_censor_days': float(aligned['llm_censor_days'].median()),
        'mean_joint_observable_lines': float(np.mean(n_obs)) if len(n_obs) else np.nan,
    })
    print(f'  {TOXICITY_DISPLAY[tox]:22s} IBS={point_est:.4f} (95% CI {ci_lower:.4f}-{ci_upper:.4f})  '
          f'agreement={agreement:.4f}  [{n_grid}/50 grid pts]  n={len(aligned):,}')

brier_df = pd.DataFrame(brier_records).sort_values('time_integrated_brier')
print(f'Brier rows: {len(brier_df)}')

## Null Model Comparison — Random Binary Endpoint

The IBS by itself is harder to interpret without knowing what level of disagreement you'd expect from a non-informative comparator. The **null model** generates a random binary endpoint `Y_null(t)` for each patient, where:

`Y_null(t) ~ Bernoulli(prevalence(t))`  and  `prevalence(t) = 1 - S_KM(t)`

This is a "no information" baseline: a random binary endpoint with the correct marginal event rate but no patient-specific information. The expected disagreement with the GS can be computed analytically:

`E[BS_null(t)] = (n_events × (1-p) + n_nonevents × p) / n`

where `p = prevalence(t)` and `n_events` = number of patients with GS event by time *t*.

**Reported metrics:**
- `null_ibs`: Expected time-integrated Brier score for the random null endpoint
- `ibs_difference`: LLM IBS − Null IBS (negative means LLM is better)
- `relative_improvement`: (Null IBS − LLM IBS) / Null IBS (positive means LLM is better)
- Bootstrap 95% CIs for all metrics

**Interpretation:** If the LLM endpoint is informative, it should disagree with the GS less often than a random endpoint would. A positive relative improvement indicates the LLM carries patient-specific information beyond population prevalence.

In [ ]:
# ===========================================================================
# NULL MODEL COMPARISON — Prevalence-Matched Random Endpoint
#
# The null model generates a random binary endpoint Y_null(t) with the SAME
# marginal event rate as the LLM (not the GS). This controls for over-calling.
#
# Y_null(t) ~ Bernoulli(p_LLM(t)) where p_LLM(t) = 1 - S_KM_LLM(t)
#
# The expected disagreement with GS is computed analytically:
#   E[BS_null(t)] = (n_gs_events * (1 - p_LLM) + n_gs_nonevents * p_LLM) / n
#
# This asks: "Given you're calling this many events, are you assigning them
# to the right patients at the right times, or is it just random?"
#
# Outputs:
#   - null_ibs: Expected IBS for random endpoint with LLM's event rate
#   - ibs_diff: LLM IBS - Null IBS (negative = LLM is better)
#   - relative_improvement: (Null IBS - LLM IBS) / Null IBS (positive = LLM is better)
#   - Bootstrap 95% CIs for all metrics
# ===========================================================================

def _integrated_brier_null_prevalence_matched(gs_times, gs_events, gs_censor,
                                               llm_times, llm_events, llm_censor,
                                               time_grid):
    """
    Prevalence-matched null model IBS.
    
    At each time t, the null endpoint Y_null ~ Bernoulli(p_LLM(t)) where p_LLM(t)
    is the LLM's marginal prevalence at t (from LLM's Kaplan-Meier curve).
    
    The expected disagreement with the GS binary status Y_GS is:
    E[(Y_null - Y_GS)²] = (n_gs_events * (1 - p_LLM) + n_gs_nonevents * p_LLM) / n
    
    This controls for over-calling: if LLM calls 5x more events than GS, the null
    also calls 5x more events. The comparison then tests whether LLM assigns
    events to the right patients/times better than random.
    """
    gs_times = np.asarray(gs_times, float)
    gs_events = np.asarray(gs_events, int)
    gs_censor = np.asarray(gs_censor, float)
    llm_times = np.asarray(llm_times, float)
    llm_events = np.asarray(llm_events, int)
    llm_censor = np.asarray(llm_censor, float)
    
    valid = (np.isfinite(gs_times) & np.isfinite(gs_censor) &
             np.isfinite(llm_times) & np.isfinite(llm_censor))
    if not valid.any():
        return np.nan
    
    gs_times, gs_events, gs_censor = gs_times[valid], gs_events[valid], gs_censor[valid]
    llm_times, llm_events, llm_censor = llm_times[valid], llm_events[valid], llm_censor[valid]
    
    # Fit KM to LLM endpoint to get LLM's marginal prevalence at each t
    kmf_llm = KaplanMeierFitter()
    kmf_llm.fit(llm_times, llm_events)
    
    bs = []
    vt = []
    for t in time_grid:
        # Observable in both arms at time t
        gs_obs = ((gs_events == 1) & (gs_times <= t)) | (gs_censor > t)
        llm_obs = ((llm_events == 1) & (llm_times <= t)) | (llm_censor > t)
        mask = gs_obs & llm_obs
        if not mask.any():
            continue
        
        # GS status at time t (binary) for observable patients
        y_gs = ((gs_events == 1) & (gs_times <= t))[mask].astype(int)
        n = len(y_gs)
        n_gs_events = y_gs.sum()
        n_gs_nonevents = n - n_gs_events
        
        # LLM's marginal prevalence at t: P_LLM(event by t) = 1 - S_LLM(t)
        p_llm = 1.0 - kmf_llm.predict(t)
        
        # Expected disagreement for random endpoint with LLM's prevalence
        # E[BS] = (n_gs_events * (1 - p_llm) + n_gs_nonevents * p_llm) / n
        expected_bs = (n_gs_events * (1 - p_llm) + n_gs_nonevents * p_llm) / n
        
        bs.append(expected_bs)
        vt.append(t)
    
    if len(bs) < 2:
        return np.nan
    
    _trapz = getattr(np, 'trapezoid', None) or np.trapz
    rng_t = vt[-1] - vt[0]
    if rng_t <= 0:
        return np.nan
    return float(max(0.0, min(1.0, _trapz(bs, vt) / rng_t)))


# Compute null model metrics for each toxicity
N_PERMUTE = 10000  # permutation test iterations for p-value

null_records = []
for tox in TOXICITY_COLUMNS:
    gs_s = gs_survival[tox]
    llm_s = llm_survival[tox].set_index(['mrn', 'lot_clean'])
    
    # Align to same set used for LLM IBS
    aligned = gs_s.set_index(['mrn', 'lot_clean']).join(
        llm_s[['llm_onset_days', 'event', 'time', 'censor_days']].rename(columns={
            'event': 'llm_event', 'time': 'llm_time', 'censor_days': 'llm_censor_days'
        }),
        how='inner'
    )
    
    # Point estimate for null model (prevalence-matched to LLM)
    null_ibs = _integrated_brier_null_prevalence_matched(
        aligned['time'].values, aligned['event'].values, aligned['censor_days'].values,
        aligned['llm_time'].values, aligned['llm_event'].values, aligned['llm_censor_days'].values,
        TIME_GRID_DAYS
    )
    
    # Get LLM IBS from previously computed results
    llm_ibs = brier_df[brier_df['toxicity'] == tox]['time_integrated_brier'].values[0]
    
    # Difference and relative improvement
    ibs_diff = llm_ibs - null_ibs  # negative = LLM is better
    rel_improvement = (null_ibs - llm_ibs) / null_ibs if null_ibs > 0 else np.nan  # positive = LLM is better
    
    # Bootstrap for CIs
    groups = aligned.index.get_level_values('mrn').values
    uniq = np.unique(groups)
    idx_by_pt = {g: np.where(groups == g)[0] for g in uniq}
    rng = np.random.default_rng(RNG_SEED)
    
    boot_null = np.empty(N_BOOTSTRAP)
    boot_llm = np.empty(N_BOOTSTRAP)
    for b in range(N_BOOTSTRAP):
        pick = rng.choice(uniq, size=len(uniq), replace=True)
        idx = np.concatenate([idx_by_pt[g] for g in pick])
        
        boot_null[b] = _integrated_brier_null_prevalence_matched(
            aligned['time'].values[idx], aligned['event'].values[idx], aligned['censor_days'].values[idx],
            aligned['llm_time'].values[idx], aligned['llm_event'].values[idx], aligned['llm_censor_days'].values[idx],
            TIME_GRID_DAYS
        )
        boot_llm[b] = _integrated_brier_endpoint_agreement(
            aligned['time'].values[idx], aligned['event'].values[idx], aligned['censor_days'].values[idx],
            aligned['llm_time'].values[idx], aligned['llm_event'].values[idx], aligned['llm_censor_days'].values[idx],
            TIME_GRID_DAYS
        )[0]
    
    boot_diff = boot_llm - boot_null
    boot_rel = (boot_null - boot_llm) / boot_null
    
    null_ci_lo, null_ci_hi = np.nanpercentile(boot_null, [2.5, 97.5])
    diff_ci_lo, diff_ci_hi = np.nanpercentile(boot_diff, [2.5, 97.5])
    rel_ci_lo, rel_ci_hi = np.nanpercentile(boot_rel, [2.5, 97.5])
    
    # Permutation test for p-value: shuffle LLM events among patients to break association
    # Under H0: LLM endpoint is no better than random assignment of events
    # Test statistic: IBS difference (LLM - Null); observed is negative if LLM is better
    perm_ibs = np.empty(N_PERMUTE)
    for p in range(N_PERMUTE):
        # Shuffle LLM event assignments among lines (breaks patient-specific information)
        perm_idx = rng.permutation(len(aligned))
        perm_ibs[p] = _integrated_brier_endpoint_agreement(
            aligned['time'].values, aligned['event'].values, aligned['censor_days'].values,
            aligned['llm_time'].values[perm_idx], aligned['llm_event'].values[perm_idx], 
            aligned['llm_censor_days'].values[perm_idx],
            TIME_GRID_DAYS
        )[0]
    
    # One-sided p-value: proportion of permuted IBS <= observed LLM IBS
    # (LLM is better if its IBS is lower, so we test if observed is unusually low)
    p_value = (np.sum(perm_ibs <= llm_ibs) + 1) / (N_PERMUTE + 1)
    
    null_records.append({
        'toxicity': tox,
        'toxicity_display': TOXICITY_DISPLAY[tox],
        'llm_ibs': llm_ibs,
        'null_ibs': null_ibs,
        'null_ibs_ci_lower': null_ci_lo,
        'null_ibs_ci_upper': null_ci_hi,
        'ibs_difference': ibs_diff,
        'ibs_diff_ci_lower': diff_ci_lo,
        'ibs_diff_ci_upper': diff_ci_hi,
        'relative_improvement': rel_improvement,
        'rel_improvement_ci_lower': rel_ci_lo,
        'rel_improvement_ci_upper': rel_ci_hi,
        'p_value_vs_null': p_value,
    })
    
    print(f'  {TOXICITY_DISPLAY[tox]:22s} LLM={llm_ibs:.4f}  Null={null_ibs:.4f}  '
          f'Diff={ibs_diff:+.4f} [{diff_ci_lo:+.4f}, {diff_ci_hi:+.4f}]  '
          f'RelImpr={100*rel_improvement:+.1f}%  p={p_value:.4f}')

null_df = pd.DataFrame(null_records)

# Merge with original brier_df and save
brier_with_null = brier_df.merge(null_df[['toxicity', 'null_ibs', 'null_ibs_ci_lower', 'null_ibs_ci_upper',
                                           'ibs_difference', 'ibs_diff_ci_lower', 'ibs_diff_ci_upper',
                                           'relative_improvement', 'rel_improvement_ci_lower', 'rel_improvement_ci_upper',
                                           'p_value_vs_null']],
                                  on='toxicity', how='left')

# Save updated CSV
NULL_CSV_OUT = os.path.join(RESULTS_DIR, 'Brier_Score_S1J1_with_null.csv')
brier_with_null.to_csv(NULL_CSV_OUT, index=False)
print(f'\nSaved: {os.path.basename(NULL_CSV_OUT)}')

## Panel S1J — Time-integrated Brier disagreement


In [ ]:
PANEL_TOX_ORDER = ['liver_toxicity', 'hypothyroidism', 'pneumonitis',
                   'colitis', 'adrenal_insufficiency', 'hyperthyroidism']
assert set(PANEL_TOX_ORDER) == set(TOXICITY_COLUMNS), 'PANEL_TOX_ORDER out of sync'

plot_df = brier_df.set_index('toxicity').loc[PANEL_TOX_ORDER].reset_index()
x = np.arange(len(plot_df))

fig, ax = plt.subplots(figsize=(2.5112, 1.8949))

yerr = np.array([
    plot_df['time_integrated_brier'] - plot_df['ci_95_lower'],
    plot_df['ci_95_upper'] - plot_df['time_integrated_brier'],
])
ax.bar(x, plot_df['time_integrated_brier'], width=0.6, color='#4C72B0',
       edgecolor='white', linewidth=0.5, yerr=yerr, capsize=3,
       error_kw={'linewidth': 0.8})

for i, v in enumerate(plot_df['time_integrated_brier']):
    ax.text(i, plot_df['ci_95_upper'].iloc[i] + 0.0015, f'{v:.3f}',
            ha='center', va='bottom', fontsize=5)

ax.set_ylabel('Time-integrated\nBrier disagreement (12mo)')
ax.set_xticks(x)
ax.set_xticklabels(plot_df['toxicity_display'], rotation=30, ha='right')
ax.set_ylim(0, max(plot_df['ci_95_upper'].max() * 1.25, 0.05))

for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)
for spine in ('bottom', 'left'):
    ax.spines[spine].set_linewidth(0.6)
ax.tick_params(width=0.6, length=3)

fig.subplots_adjust(left=0.28, right=0.97, top=0.95, bottom=0.32)
fig.savefig(BRIER_PDF_OUT, dpi=450)
print(f'Saved: {os.path.basename(BRIER_PDF_OUT)}')
plt.show()


## Onset agreement — all six toxicities

Conditioned on lines the GS flagged. `medΔd` is median days between GS and LLM onset among lines
where both flagged; negative means the pipeline flagged earlier.

In [ ]:

def build_detection_frame(tox):
    """One row per GS-positive line: GS onset, and LLM onset (or censored if never flagged)."""
    g = gs_survival[tox]
    g = g[g['event'] == 1][['mrn', 'lot_clean', 'gs_onset_days', 'censor_days']].copy()
    l = llm_survival[tox][['mrn', 'lot_clean', 'llm_onset_days']]
    d = g.merge(l, on=['mrn', 'lot_clean'], how='left')
    d['llm_detected'] = d['llm_onset_days'].notna().astype(int)
    d['llm_time'] = np.where(d['llm_detected'] == 1, d['llm_onset_days'], d['censor_days'])
    d['onset_diff'] = d['llm_onset_days'] - d['gs_onset_days']
    return d

print(f'{"Toxicity":<24} {"GSev":>6} {"detected":>9} {"sens":>6} '
      f'{"medΔd":>7} {"±30d":>6} {"±60d":>6}')
print('-' * 70)
detection = {}
det_rows = []
for tox in TOXICITY_COLUMNS:
    d = build_detection_frame(tox)
    detection[tox] = d
    both = d[d['llm_detected'] == 1]
    sens = d['llm_detected'].mean()
    med = both['onset_diff'].median() if len(both) else np.nan
    w30 = (both['onset_diff'].abs() <= 30).mean() if len(both) else np.nan
    w60 = (both['onset_diff'].abs() <= 60).mean() if len(both) else np.nan
    print(f'{TOXICITY_DISPLAY[tox]:<24} {len(d):>6,} {len(both):>9,} {100*sens:>5.1f}% '
          f'{med:>7.0f} {100*w30:>5.1f}% {100*w60:>5.1f}%')
    det_rows.append({'toxicity': tox, 'toxicity_display': TOXICITY_DISPLAY[tox],
                     'n_gs_events': len(d), 'n_detected': len(both), 'sensitivity': sens,
                     'median_onset_diff_days': med,
                     'pct_within_30d': w30, 'pct_within_60d': w60})

detection_df = pd.DataFrame(det_rows)
detection_df.to_csv(KM_CSV_OUT, index=False)
print(f'\nSaved: {os.path.basename(KM_CSV_OUT)}')

## Median timing error among concordant events

Restricted to lines both arms flagged (same set as S1K). **Error** is `|LLM onset − GS onset|` in days, with a bootstrap 95% CI. Signed difference (`LLM − GS`) is bias: negative means the pipeline flagged earlier.

The **pooled** row is the one estimate across six irAEs. Per-toxicity rows are for the supplement; hyperthyroidism is n=8 so its CI will be wide.

In [ ]:
_SEED = RNG_SEED if 'RNG_SEED' in dir() else 0


def _boot_median_ci(x, n_boot=2000, seed=_SEED):
    """Percentile bootstrap CI for the median. Unit = one concordant (patient, LOT)."""
    x = np.asarray(x, float)
    x = x[np.isfinite(x)]
    if len(x) == 0:
        return np.nan, np.nan, np.nan
    rng = np.random.default_rng(seed)
    meds = np.array([np.median(rng.choice(x, size=len(x), replace=True))
                     for _ in range(n_boot)])
    lo, hi = np.percentile(meds, [2.5, 97.5])
    return float(np.median(x)), float(lo), float(hi)


def _timing_row(label, diffs, n_gs=np.nan):
    diffs = np.asarray(diffs, float)
    diffs = diffs[np.isfinite(diffs)]
    abs_d = np.abs(diffs)
    med_abs, abs_lo, abs_hi = _boot_median_ci(abs_d)
    med_sgn, sgn_lo, sgn_hi = _boot_median_ci(diffs)
    return {
        'toxicity': label,
        'n_concordant': int(len(diffs)),
        'n_gs_events': n_gs,
        'median_abs_error_days': med_abs,
        'median_abs_error_days_ci_lo': abs_lo,
        'median_abs_error_days_ci_hi': abs_hi,
        'median_abs_error_weeks': med_abs / 7.0,
        'median_abs_error_weeks_ci_lo': abs_lo / 7.0,
        'median_abs_error_weeks_ci_hi': abs_hi / 7.0,
        'median_signed_diff_days': med_sgn,          # LLM − GS; negative = LLM earlier
        'median_signed_diff_days_ci_lo': sgn_lo,
        'median_signed_diff_days_ci_hi': sgn_hi,
        'pct_llm_earlier': float((diffs < 0).mean()) if len(diffs) else np.nan,
        'pct_within_7d': float((abs_d <= 7).mean()) if len(diffs) else np.nan,
        'pct_within_14d': float((abs_d <= 14).mean()) if len(diffs) else np.nan,
        'pct_within_30d': float((abs_d <= 30).mean()) if len(diffs) else np.nan,
    }


timing_rows = []
pooled = []
for tox in TOXICITY_COLUMNS:
    both = detection[tox]
    both = both.loc[both['llm_detected'] == 1, 'onset_diff']
    pooled.append(both)
    timing_rows.append(_timing_row(tox, both, n_gs=len(detection[tox])))

timing_rows.append(_timing_row('pooled_six_iraes', pd.concat(pooled, ignore_index=True),
                               n_gs=int(sum(len(detection[t]) for t in TOXICITY_COLUMNS))))
timing_df = pd.DataFrame(timing_rows)
timing_df['toxicity_display'] = timing_df['toxicity'].map(
    lambda t: 'Pooled (six irAEs)' if t == 'pooled_six_iraes' else TOXICITY_DISPLAY[t])
timing_df.to_csv(TIMING_CSV_OUT, index=False)

def _fmt_days(row, prefix):
    return (f"{row[f'{prefix}']:.0f} days "
            f"(95% CI {row[f'{prefix}_ci_lo']:.0f}–{row[f'{prefix}_ci_hi']:.0f})")

def _fmt_signed_days(row):
    return (f"{row['median_signed_diff_days']:.2f} days "
            f"(95% CI {row['median_signed_diff_days_ci_lo']:.2f} to "
            f"{row['median_signed_diff_days_ci_hi']:.2f})")

def _fmt_weeks(row):
    return (f"{row['median_abs_error_weeks']:.1f} weeks "
            f"(95% CI {row['median_abs_error_weeks_ci_lo']:.1f}–"
            f"{row['median_abs_error_weeks_ci_hi']:.1f})")

print(f'{"Toxicity":<24} {"n":>5}  median |error| (95% CI)                 median signed LLM−GS')
print('-' * 100)
for _, r in timing_df.iterrows():
    print(f"{r['toxicity_display']:<24} {r['n_concordant']:>5,}  "
          f"{_fmt_days(r, 'median_abs_error_days'):<40s}  "
          f"{_fmt_signed_days(r)}")

pool = timing_df[timing_df['toxicity'] == 'pooled_six_iraes'].iloc[0]
print(f'\nSaved: {os.path.basename(TIMING_CSV_OUT)}')
print(
    f"Among {int(pool['n_concordant']):,} lines where both the gold standard and the pipeline "
    f"flagged the same irAE, the median absolute onset error was "
    f"{_fmt_days(pool, 'median_abs_error_days')} "
    f"[{_fmt_weeks(pool)}]. "
    f"The signed median (LLM − GS) was {_fmt_signed_days(pool)} "
    f"({100 * pool['pct_llm_earlier']:.0f}% of concordant events had an earlier pipeline date)."
)

## Panel S1K — onset timing, concordant events only

Both arms restricted to the same lines, so the curves compare timing and nothing else.

**Reporting notes.**  Wilcoxon P is uncorrected; with six toxicities the Bonferroni threshold is
0.0083, so a P near 0.03 does **not** support a systematic "detects earlier" claim. The defensible
statement is close agreement with no meaningful offset.

In [ ]:
FEATURE_TOX = 'adrenal_insufficiency'
assert FEATURE_TOX in TOXICITY_COLUMNS

d_both = detection[FEATURE_TOX]
d_both = d_both[d_both['llm_detected'] == 1].copy()
n_both = len(d_both)
n_gs = len(detection[FEATURE_TOX])
sens = n_both / n_gs

_ev = np.ones(n_both)
kmf_gs = KaplanMeierFitter().fit(d_both['gs_onset_days'] / 30.44, _ev, label='Gold Standard')
kmf_llm = KaplanMeierFitter().fit(d_both['llm_onset_days'] / 30.44, _ev, label='LLM Pipeline')

# Paired test on the same lines (Wilcoxon signed-rank; paired by construction)
from scipy.stats import wilcoxon
_diff = d_both['onset_diff'].dropna()
_stat, _p = wilcoxon(_diff) if len(_diff) > 10 else (np.nan, np.nan)
_med = _diff.median()
_q1, _q3 = _diff.quantile([.25, .75])

fig, ax = plt.subplots(figsize=(2.5112, 1.8949))
for kmf, color, ls, lab in [(kmf_gs, '#1f77b4', '-', 'Gold Standard'),
                            (kmf_llm, '#ff7f0e', '--', 'LLM Pipeline')]:
    sf = kmf.survival_function_
    ax.step(sf.index, (1 - sf.iloc[:, 0]) * 100, where='post',
            color=color, linewidth=1.0, linestyle=ls, label=lab)

ax.set_title(f'{TOXICITY_DISPLAY[FEATURE_TOX]}', fontsize=6, pad=2)
ax.set_xlim(0, T_MONTHS)
ax.set_ylim(0, 105)
ax.set_ylabel('Cumulative onset (%)')
ax.set_xlabel('Months from line start')

for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)
for spine in ('bottom', 'left'):
    ax.spines[spine].set_linewidth(0.6)
ax.tick_params(width=0.6, length=3)
ax.legend(loc='upper left', fontsize=5, frameon=False, handlelength=1.4,
          handletextpad=0.4, labelspacing=0.3, borderaxespad=0.2)

fig.subplots_adjust(left=0.28, right=0.97, top=0.90, bottom=0.32)
_out = os.path.join(RESULTS_DIR, 'KM_Concordant_S1K.pdf')
fig.savefig(_out, dpi=450)
print(f'Saved: {os.path.basename(_out)}')
print(f'  n concordant = {n_both:,} / {n_gs:,} GS events ({100*sens:.1f}% sensitivity)')
print(f'  median onset difference = {_med:+.0f}d (IQR {_q1:+.0f} to {_q3:+.0f}), Wilcoxon P = {_p:.2e}')
print(f'  LLM earlier in {100*(_diff < 0).mean():.1f}% of concordant events')
plt.show()